# 07 — Cloud Embedders: OpenAI, Cohere, Gemini

Medha is **embedder-agnostic**: you can swap the default local FastEmbed model
for any cloud embedding API without changing a single line of search/store code.

This notebook covers three cloud embedders:
1. **OpenAI** (`text-embedding-3-small` / `text-embedding-3-large`) — industry standard
2. **Cohere** (`embed-english-v3.0`) — strong multilingual + auto input-type routing
3. **Gemini** (`text-embedding-004`) — Google, 768 dims, internal chunking

**Requirements:**
```bash
pip install "medha-archai[openai]"   # OpenAI
pip install "medha-archai[cohere]"   # Cohere
pip install "medha-archai[gemini]"   # Gemini
```


## Cell 1: Imports & API Key

In [ ]:
import os
import time

from medha import Medha, Settings
from medha.embeddings.openai_adapter import OpenAIAdapter
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

# The adapter reads OPENAI_API_KEY automatically if set in the environment.
# You can also pass it explicitly: OpenAIAdapter(api_key="sk-...")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if not OPENAI_API_KEY:
    print("WARNING: OPENAI_API_KEY not set. Cells that call the OpenAI API will fail.")
    print("Set it with: export OPENAI_API_KEY=sk-...")
else:
    print(f"API key found: sk-...{OPENAI_API_KEY[-4:]}")

## Cell 2: Embedder Comparison — FastEmbed vs OpenAI

| | FastEmbedAdapter | OpenAIAdapter |
|---|---|---|
| **Compute** | Local (CPU/GPU) | Remote API |
| **Model size** | ~80 MB (BAAI/bge-small-en-v1.5) | Server-side |
| **Latency** | ~1–5ms (after JIT warm-up) | ~50–200ms (network) |
| **Cost** | Free | ~$0.02/1M tokens |
| **Dimension** | 384 | 1536 (small), 3072 (large) |
| **Quality** | Good for English | State-of-the-art |
| **Offline** | ✅ | ❌ |
| **Rate limits** | None | Yes (mitigated by retry) |

**When to choose OpenAI:**
- You need maximum retrieval quality
- Your queries contain complex domain language or mixed languages
- You already use the OpenAI API and want a unified billing model

**When to choose FastEmbed:**
- Latency-sensitive production environments
- Air-gapped or offline deployments
- High-volume workloads where API cost matters

## Cell 3: Initialize OpenAIAdapter

Three OpenAI models are supported out of the box:
- `text-embedding-3-small` (1536 dims, cheapest, recommended default)
- `text-embedding-3-large` (3072 dims, highest quality)
- `text-embedding-ada-002` (1536 dims, legacy)

The `text-embedding-3-*` models also support **dimension reduction**: you can request
fewer dimensions (e.g., 512) to reduce storage and search cost with minimal quality loss.

In [ ]:
# Default: text-embedding-3-small, 1536 dimensions
embedder_small = OpenAIAdapter(model_name="text-embedding-3-small")
print(f"Model:     {embedder_small.model_name}")
print(f"Dimension: {embedder_small.dimension}")

# With dimension reduction: 512 dims (~66% smaller index, <5% quality loss)
embedder_compact = OpenAIAdapter(
    model_name="text-embedding-3-small",
    dimensions=512,
)
print(f"\nCompact model: {embedder_compact.model_name} (reduced to {embedder_compact.dimension} dims)")

## Cell 4: Store and Search — Drop-in Replacement

The `Medha` API is **identical** regardless of which embedder you use.
You only change the embedder at construction time.

In [ ]:
pairs = [
    ("How many customers do we have?",     "SELECT COUNT(*) FROM customers"),
    ("List top 5 products by revenue",     "SELECT product_id, SUM(amount) AS revenue FROM orders GROUP BY product_id ORDER BY revenue DESC LIMIT 5"),
    ("Average order value this month",     "SELECT AVG(total) FROM orders WHERE DATE_FORMAT(created_at,'%Y-%m') = DATE_FORMAT(NOW(),'%Y-%m')"),
    ("Customers with no orders",           "SELECT * FROM customers WHERE id NOT IN (SELECT DISTINCT customer_id FROM orders)"),
    ("Revenue by product category",        "SELECT c.name, SUM(o.amount) FROM orders o JOIN products p ON o.product_id=p.id JOIN categories c ON p.category_id=c.id GROUP BY c.name"),
]

# ---- Using OpenAI embedder ----
settings = Settings(backend_type="memory")
medha = Medha("openai_demo", embedder=embedder_small, settings=settings)
await medha.start()

print("Storing pairs with OpenAI embedder...")
for question, sql in pairs:
    await medha.store(question, sql)
print(f"  {len(pairs)} entries stored\n")

# Search — same API as any other embedder
test_questions = [
    "What is the total number of clients?",     # semantic: customers → clients
    "Top 5 products by sales",                  # semantic: revenue → sales
    "Show me customers who never placed an order",  # semantic paraphrase
]

print("Searching...")
for q in test_questions:
    start = time.perf_counter()
    hit = await medha.search(q)
    elapsed = (time.perf_counter() - start) * 1000
    print(f"  Q: {q!r}")
    print(f"     strategy={hit.strategy}, confidence={hit.confidence:.4f}, time={elapsed:.1f}ms")
    if hit.generated_query:
        print(f"     → {hit.generated_query[:80]}")
    print()

await medha.close()

## Cell 5: Dimension Reduction

`text-embedding-3-small` with `dimensions=512` stores vectors that are 3× smaller
than the full 1536-dimensional output. Qdrant retrieves them faster and uses less RAM.

OpenAI applies Matryoshka Representation Learning internally: the first N dimensions
of a full vector are a high-quality representation at dimension N.

In [ ]:
configs = [
    ("3-small full (1536d)",   OpenAIAdapter(model_name="text-embedding-3-small")),
    ("3-small compact (512d)", OpenAIAdapter(model_name="text-embedding-3-small", dimensions=512)),
    ("3-small tiny (256d)",    OpenAIAdapter(model_name="text-embedding-3-small", dimensions=256)),
]

test_q   = "How many active users are there?"
stored_q = "Total active user count"
stored_sql = "SELECT COUNT(*) FROM users WHERE status='active'"

print(f"Question stored: {stored_q!r}")
print(f"Question searched: {test_q!r}")
print()

for label, emb in configs:
    settings = Settings(backend_type="memory")
    medha = Medha(f"dim_{emb.dimension}", embedder=emb, settings=settings)
    await medha.start()
    await medha.store(stored_q, stored_sql)

    hit = await medha.search(test_q)
    print(f"  {label:30s}  strategy={hit.strategy.value:18s}  confidence={hit.confidence:.4f}")
    await medha.close()

## Cell 6: Cost Estimation

OpenAI charges per token. With Medha caching, you pay for embedding tokens at
store time (once per unique question) and at search time only for questions not
served from the L1 cache or template tier.

This cell estimates the monthly embedding cost for a typical workload.

In [ ]:
# OpenAI pricing (per 1M tokens) — check https://openai.com/pricing for latest
PRICING = {
    "text-embedding-3-small": 0.020,   # $0.020 / 1M tokens
    "text-embedding-3-large": 0.130,   # $0.130 / 1M tokens
    "text-embedding-ada-002": 0.100,   # $0.100 / 1M tokens  (legacy)
}

# Workload assumptions
MONTHLY_QUERIES    = 100_000       # total user queries per month
AVG_TOKENS_PER_Q   = 15            # average question length in tokens
CACHE_SIZE         = 500           # unique questions in the cache
HIT_RATE           = 0.75          # 75% of queries served from cache (no embedding needed)

store_tokens   = CACHE_SIZE * AVG_TOKENS_PER_Q           # one-time population cost
search_tokens  = MONTHLY_QUERIES * (1 - HIT_RATE) * AVG_TOKENS_PER_Q  # monthly misses
total_tokens   = store_tokens + search_tokens

print("Monthly Embedding Cost Estimate")
print(f"  Monthly queries      : {MONTHLY_QUERIES:,}")
print(f"  Cache hit rate       : {HIT_RATE*100:.0f}%")
print(f"  Store tokens (once)  : {store_tokens:,}")
print(f"  Search tokens/month  : {search_tokens:,.0f}  ({MONTHLY_QUERIES*(1-HIT_RATE):,.0f} misses × {AVG_TOKENS_PER_Q} tokens)")
print(f"  Total tokens/month   : {total_tokens:,.0f}")
print()
print(f"  {'Model':<30s}  {'Cost/month':>12s}  {'Without Medha':>15s}  {'Savings':>10s}")
print(f"  {'-'*30}  {'-'*12}  {'-'*15}  {'-'*10}")

for model, price_per_m in PRICING.items():
    cost_with_medha    = total_tokens / 1_000_000 * price_per_m
    cost_without_medha = MONTHLY_QUERIES * AVG_TOKENS_PER_Q / 1_000_000 * price_per_m
    savings            = cost_without_medha - cost_with_medha
    savings_pct        = savings / cost_without_medha * 100 if cost_without_medha > 0 else 0
    print(f"  {model:<30s}  ${cost_with_medha:>10.4f}  ${cost_without_medha:>13.4f}  {savings_pct:>9.1f}%")

## Cell 7: Migrating Between Embedders

If you switch embedder (e.g., from FastEmbed to OpenAI), the existing vector
collection **must be rebuilt** — vectors from different models are not comparable.

The recommended migration strategy:
1. Export your question-query pairs to a JSON file
2. Create a new Medha collection with the new embedder
3. `warm_from_file()` to re-populate
4. Retire the old collection

In [ ]:
import json
import tempfile
import os

golden_dataset = [
    {"question": "How many users?",          "generated_query": "SELECT COUNT(*) FROM users"},
    {"question": "Show all active sessions", "generated_query": "SELECT * FROM sessions WHERE active=1"},
    {"question": "Monthly revenue",          "generated_query": "SELECT SUM(amount) FROM payments WHERE MONTH(created_at)=MONTH(NOW())"},
]

# Step 1: Simulate an existing cache built with FastEmbed
print("Step 1 — Original cache (FastEmbed)")
old_embedder = FastEmbedAdapter()
async with Medha("old_collection", embedder=old_embedder, settings=Settings(backend_type="memory")) as m:
    await m.store_batch(golden_dataset)
    hit = await m.search("Total number of users")
    print(f"  Hit: {hit.strategy} — {hit.generated_query}")

# Step 2: Re-build with OpenAI embedder
print("\nStep 2 — Migrated cache (OpenAI text-embedding-3-small)")
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    json.dump(golden_dataset, f)
    export_file = f.name

new_embedder = OpenAIAdapter(model_name="text-embedding-3-small")
async with Medha("new_collection", embedder=new_embedder, settings=Settings(backend_type="memory")) as m:
    n = await m.warm_from_file(export_file)
    print(f"  Loaded {n} entries from export")
    hit = await m.search("Total number of users")
    print(f"  Hit: {hit.strategy} — {hit.generated_query}")

os.unlink(export_file)
print("\nMigration complete. Old collection can now be retired.")

## Cohere Embeddings

`CohereAdapter` wraps the Cohere Embed v3 API. It automatically sets
`input_type="search_document"` when storing and `input_type="search_query"`
when searching — no manual configuration required.

```bash
pip install "medha-archai[cohere]"
```

**Note:** You need a Cohere API key. Set `COHERE_API_KEY` in your environment
or pass it directly to `CohereAdapter(api_key=...)`.


In [ ]:
import os

from medha import Medha, Settings

try:
    from medha import CohereAdapter
    HAS_COHERE = True
    print("cohere package is installed")
except ImportError:
    HAS_COHERE = False
    print("cohere not found — install with: pip install medha-archai[cohere]")

COHERE_API_KEY = os.getenv("COHERE_API_KEY", "")
if not COHERE_API_KEY:
    print("WARNING: COHERE_API_KEY not set. Cohere cells will be skipped.")

CAN_COHERE = HAS_COHERE and bool(COHERE_API_KEY)

if CAN_COHERE:
    cohere_embedder = CohereAdapter(
        api_key=COHERE_API_KEY,
        model="embed-english-v3.0",  # 1024 dims
    )
    print(f"Model     : {cohere_embedder.model_name}")
    print(f"Dimension : {cohere_embedder.dimension}")
    print("input_type: set automatically (search_document on store, search_query on search)")

    pairs_cohere = [
        ("How many customers do we have?",  "SELECT COUNT(*) FROM customers"),
        ("List top 5 products by revenue",  "SELECT product_id, SUM(amount) FROM orders GROUP BY product_id ORDER BY 2 DESC LIMIT 5"),
        ("Average order value this month",  "SELECT AVG(total) FROM orders WHERE DATE_FORMAT(created_at,'%Y-%m') = DATE_FORMAT(NOW(),'%Y-%m')"),
    ]

    settings = Settings(backend_type="memory")
    async with Medha("cohere_demo", embedder=cohere_embedder, settings=settings) as medha:
        for q, sql in pairs_cohere:
            await medha.store(q, sql)

        hit = await medha.search("Total number of clients")
        print(f"\nSearch: strategy={hit.strategy.value}  confidence={hit.confidence:.4f}")
        print(f"Query : {hit.generated_query}")
else:
    print("Skipping — COHERE_API_KEY not set.")


## Gemini Embeddings

`GeminiAdapter` wraps the Google Generative AI Embeddings API (`text-embedding-004`).
The adapter batches requests internally in chunks of 100 to stay within the API
rate limits — no manual batching required.

```bash
pip install "medha-archai[gemini]"
```

**Note:** You need a Google AI API key. Set `GEMINI_API_KEY` in your environment
or pass it directly to `GeminiAdapter(api_key=...)`.


In [ ]:
import os

try:
    from medha import GeminiAdapter
    HAS_GEMINI = True
    print("google-generativeai package is installed")
except ImportError:
    HAS_GEMINI = False
    print("google-generativeai not found — install with: pip install medha-archai[gemini]")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    print("WARNING: GEMINI_API_KEY not set. Gemini cells will be skipped.")

CAN_GEMINI = HAS_GEMINI and bool(GEMINI_API_KEY)

if CAN_GEMINI:
    gemini_embedder = GeminiAdapter(
        api_key=GEMINI_API_KEY,
        model="text-embedding-004",  # 768 dims
    )
    print(f"Model     : {gemini_embedder.model_name}")
    print(f"Dimension : {gemini_embedder.dimension}")
    print("Batching  : internal chunking of 100 per API call")

    pairs_gemini = [
        ("How many customers do we have?",  "SELECT COUNT(*) FROM customers"),
        ("List top 5 products by revenue",  "SELECT product_id, SUM(amount) FROM orders GROUP BY product_id ORDER BY 2 DESC LIMIT 5"),
        ("Average order value this month",  "SELECT AVG(total) FROM orders WHERE DATE_FORMAT(created_at,'%Y-%m') = DATE_FORMAT(NOW(),'%Y-%m')"),
    ]

    settings = Settings(backend_type="memory")
    async with Medha("gemini_demo", embedder=gemini_embedder, settings=settings) as medha:
        for q, sql in pairs_gemini:
            await medha.store(q, sql)

        hit = await medha.search("Total number of clients")
        print(f"\nSearch: strategy={hit.strategy.value}  confidence={hit.confidence:.4f}")
        print(f"Query : {hit.generated_query}")
else:
    print("Skipping — GEMINI_API_KEY not set.")


## Embedder Comparison

| Embedder | Class | Dimension | Batch size | Rate limit | Cost |
|---|---|---|---|---|---|
| FastEmbed (local) | `FastEmbedAdapter` | 384 | 256 | None | Free |
| OpenAI small | `OpenAIAdapter` | 1536 | 2048 | 3 000 RPM | $0.020/1M tokens |
| OpenAI large | `OpenAIAdapter` | 3072 | 2048 | 3 000 RPM | $0.130/1M tokens |
| Cohere v3 | `CohereAdapter` | 1024 | 96 | 10 000 RPM | $0.100/1M tokens |
| Gemini 004 | `GeminiAdapter` | 768 | 100¹ | 1 500 RPM | Free tier available |

¹ `GeminiAdapter` splits large batches into chunks of 100 internally.

**Choosing an embedder:**
- **No cloud dependency / offline:** `FastEmbedAdapter` — local, free, 384 dims
- **Highest quality English:** `OpenAIAdapter` (`text-embedding-3-large`, 3072 dims)
- **Multilingual + automatic input-type routing:** `CohereAdapter`
- **Google stack / generous free quota:** `GeminiAdapter` (`text-embedding-004`)
- **High-dimensional vectors (≥ 1024 dims):** Cohere or OpenAI large + `VectorChordBackend`
